# GTE-Qwen2-7B-Instruct → Qdrant — bản ổn định cho Kaggle T4

Pipeline này dùng trực tiếp `AutoModel` và `AutoTokenizer`, không phụ thuộc `sentence-transformers`, SciPy hay scikit-learn.

- Bắt buộc custom GTE với `trust_remote_code=True`.
- Document được encode nguyên văn.
- Query có dạng `Instruct: ...\nQuery: ...`.
- Last-token pooling và L2 normalization.
- Vector 3584 chiều, cosine distance.
- Không fallback sang Qwen2 gốc vì vector sẽ không tương thích khi test fusion.

## 0. Cài bộ thư viện tối thiểu

Chạy cell này trong session Kaggle mới, sau đó chọn **Restart Session**. `--no-deps` giúp không làm thay đổi NumPy/SciPy có sẵn của Kaggle.

In [1]:
!pip -q install --no-deps --force-reinstall \
    "huggingface-hub==0.24.7" \
    "tokenizers==0.19.1" \
    "transformers==4.44.2" \
    "accelerate==0.33.0" \
    "bitsandbytes==0.49.2"
!pip -q install -U "qdrant-client>=1.10,<2"

print("Cài đặt xong. Hãy Restart Session rồi chạy lại từ cell tiếp theo.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.5/417.5 kB 11.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 65.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 100.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 7.3 MB/s eta 0:00:00:00:01
Cài đặt xong. Hãy Restart Session rồi chạy lại từ cell tiếp theo.


## 1. Kiểm tra môi trường, Qdrant và corpus

In [2]:
import gc
import glob
import json
import os

import accelerate
import bitsandbytes
import torch
import transformers
from kaggle_secrets import UserSecretsClient
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams
from transformers import AutoConfig, AutoModel, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Alibaba-NLP/gte-Qwen2-7B-instruct"
MODEL_REVISION = "a8d08b36ada9cacfe34c4d6f80957772a025daf2"
COLLECTION = "laws_gte_qwen2"
EXPECTED_DIM = 3584
MAX_LENGTH = 8192
ENCODE_BATCH = 1
UPSERT_BATCH = 64
INSTRUCTION = "Given a legal question in Vietnamese, retrieve the most relevant law articles"

EXPECTED_VERSIONS = {
    "transformers": "4.44.2",
    "accelerate": "0.33.0",
    "bitsandbytes": "0.49.2",
}
actual_versions = {
    "transformers": transformers.__version__,
    "accelerate": accelerate.__version__,
    "bitsandbytes": bitsandbytes.__version__,
}
print("Versions:", actual_versions)
assert actual_versions == EXPECTED_VERSIONS, (
    "Sai phiên bản. Hãy Restart Session sau cell cài đặt. "
    f"Expected={EXPECTED_VERSIONS}, actual={actual_versions}"
)
assert hasattr(transformers.DynamicCache, "get_usable_length"), (
    "Cache API không đúng transformers 4.44.2. Hãy Restart Session."
)
assert torch.cuda.is_available(), "Hãy bật GPU: Settings > Accelerator > GPU."
print("CUDA:", torch.version.cuda, "| GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(" ", i, torch.cuda.get_device_name(i))

secrets = UserSecretsClient()
qdrant_url = secrets.get_secret("QDRANT_URL")
try:
    qdrant_key = secrets.get_secret("QDRANT_KEY")
except Exception:
    qdrant_key = secrets.get_secret("QDRANT_API_KEY")
assert qdrant_url and qdrant_key, "Thiếu QDRANT_URL hoặc QDRANT_KEY/QDRANT_API_KEY."

qc = QdrantClient(url=qdrant_url, api_key=qdrant_key, timeout=300)
print("Qdrant collections:", [x.name for x in qc.get_collections().collections])

def find_corpus():
    candidates = [
        "/kaggle/working/corpus_law_pub.json",
        "corpus_law_pub.json",
    ]
    candidates += glob.glob("/kaggle/input/**/corpus_law_pub.json", recursive=True)
    for path in candidates:
        if os.path.isfile(path):
            return path
    raise FileNotFoundError("Không tìm thấy corpus_law_pub.json.")

corpus_path = find_corpus()
with open(corpus_path, encoding="utf-8") as f:
    corpus = json.load(f)

records = []
for law in corpus:
    for article_no, article in enumerate(law["content"], start=1):
        text = (article.get("content_Article") or "").strip()
        if not text:
            continue
        aid = int(article["aid"])
        records.append({
            "point_id": aid,
            "aid": aid,
            "law_id": law["law_id"],
            "article_no": article_no,
            "text": text,
        })

assert records, "Corpus rỗng."
assert len({r['aid'] for r in records}) == len(records), "aid bị trùng."
print("Corpus:", corpus_path, "| laws:", len(corpus), "| articles:", len(records))

Versions: {'transformers': '4.44.2', 'accelerate': '0.33.0', 'bitsandbytes': '0.49.2'}
CUDA: 12.8 | GPUs: 2
  0 Tesla T4
  1 Tesla T4
Qdrant collections: ['alqac_laws_raw', 'alqac_laws_test']
Corpus: /kaggle/input/datasets/hieu2k5/corpus-law/corpus_law_pub.json | laws: 18 | articles: 3352


## 2. Load custom GTE 4-bit và kiểm tra forward

In [3]:
# T4 không dùng FlashAttention 2. Remote repository chỉ xem flash_attn là tùy chọn.
import transformers.dynamic_module_utils as dynamic_utils
if not getattr(dynamic_utils, "_gte_ignore_optional_flash_attn", False):
    _original_get_imports = dynamic_utils.get_imports
    dynamic_utils.get_imports = lambda filename: [
        name for name in _original_get_imports(filename) if name != "flash_attn"
    ]
    dynamic_utils._gte_ignore_optional_flash_attn = True

def clear_cuda():
    gc.collect()
    torch.cuda.empty_cache()
    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass

# Kiểm tra bitsandbytes NF4 trước khi tải model lớn.
_bnb_test = bitsandbytes.nn.Linear4bit(
    16,
    16,
    bias=False,
    compute_dtype=torch.float16,
    quant_type="nf4",
).to("cuda:0")
with torch.inference_mode():
    _bnb_output = _bnb_test(torch.randn(2, 16, device="cuda:0", dtype=torch.float16))
assert bool(torch.isfinite(_bnb_output).all()), "bitsandbytes NF4 tạo NaN/Inf."
del _bnb_test, _bnb_output
clear_cuda()
print("bitsandbytes NF4 smoke test: OK")

# Accelerate 0.33 có thể gọi model.to() khi device_map chỉ còn một GPU; model 4-bit
# không cho phép thao tác đó. force_hooks giữ việc dispatch đúng cho cả một và hai T4.
import accelerate.big_modeling as accelerate_big_modeling
import transformers.modeling_utils as transformers_modeling_utils
if not getattr(accelerate_big_modeling, "_gte_4bit_dispatch_safe", False):
    _original_dispatch_model = accelerate_big_modeling.dispatch_model
    def _dispatch_model_4bit_safe(model_to_dispatch, *args, **kwargs):
        if getattr(model_to_dispatch, "is_loaded_in_4bit", False):
            kwargs["force_hooks"] = True
        return _original_dispatch_model(model_to_dispatch, *args, **kwargs)
    accelerate_big_modeling.dispatch_model = _dispatch_model_4bit_safe
    transformers_modeling_utils.dispatch_model = _dispatch_model_4bit_safe
    accelerate_big_modeling._gte_4bit_dispatch_safe = True

config = AutoConfig.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=True,
)
# Guard cho schema rope: remote modeling_qwen.py bắt buộc config.rope_theta.
if not hasattr(config, "rope_theta"):
    rope_data = getattr(config, "rope_parameters", None) or getattr(config, "rope_scaling", None) or {}
    config.rope_theta = float(rope_data.get("rope_theta", rope_data.get("base", 1_000_000.0)))
config.use_cache = False
assert hasattr(config, "rope_theta"), "Không thiết lập được config.rope_theta."

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=True,
)
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

try:
    model = AutoModel.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        trust_remote_code=True,
        config=config,
        quantization_config=quantization,
        device_map="auto",
        attn_implementation="eager",
        low_cpu_mem_usage=True,
    )
except Exception as exc:
    raise RuntimeError(
        "Không load được custom GTE. Không fallback trust_remote_code=False. "
        "Hãy chắc chắn đã Restart Session. Lỗi gốc: " + repr(exc)
    ) from exc

model.eval()
model.config.use_cache = False
model_module = type(model).__module__
hidden_size = int(model.config.hidden_size)
print("Model module:", model_module)
print("Hidden size:", hidden_size, "| rope_theta:", model.config.rope_theta)
print("Device map:", getattr(model, "hf_device_map", None))
assert "transformers_modules" in model_module, "Không phải custom GTE architecture."
assert getattr(model, "is_loaded_in_4bit", False), "Model chưa load 4-bit."
assert hidden_size == EXPECTED_DIM

def last_token_pool(last_hidden_state, attention_mask):
    attention_mask = attention_mask.to(last_hidden_state.device)
    if bool((attention_mask[:, -1].sum() == attention_mask.shape[0]).item()):
        return last_hidden_state[:, -1]
    sequence_lengths = attention_mask.sum(dim=1) - 1
    rows = torch.arange(last_hidden_state.shape[0], device=last_hidden_state.device)
    return last_hidden_state[rows, sequence_lengths]

def encode_texts(texts, batch_size=ENCODE_BATCH):
    if not texts:
        return torch.empty((0, EXPECTED_DIM), dtype=torch.float32)
    batch_size = max(1, int(batch_size))
    while True:
        try:
            output_vectors = []
            input_device = model.get_input_embeddings().weight.device
            for start in range(0, len(texts), batch_size):
                batch_texts = [(text or " ").strip() or " " for text in texts[start:start + batch_size]]
                inputs = tokenizer(
                    batch_texts,
                    max_length=MAX_LENGTH,
                    padding=True,
                    truncation=True,
                    return_tensors="pt",
                )
                inputs = {key: value.to(input_device) for key, value in inputs.items()}
                with torch.inference_mode():
                    outputs = model(**inputs, use_cache=False, return_dict=True)
                    vectors = last_token_pool(outputs.last_hidden_state, inputs["attention_mask"])
                    vectors = torch.nn.functional.normalize(vectors.float(), p=2, dim=1)
                output_vectors.append(vectors.cpu())
                done = min(start + batch_size, len(texts))
                if done % 100 == 0 or done == len(texts):
                    print("embedded", done, "/", len(texts))
                del inputs, outputs, vectors
            return torch.cat(output_vectors, dim=0)
        except torch.cuda.OutOfMemoryError:
            clear_cuda()
            if batch_size == 1:
                raise
            batch_size = max(1, batch_size // 2)
            print("CUDA OOM, retry batch_size=", batch_size)

def make_query(text):
    return f"Instruct: {INSTRUCTION}\nQuery: {(text or '').strip()}"

probe = encode_texts([records[0]["text"], make_query("bồi thường do súc vật gây ra")], 1)
probe_norms = torch.linalg.vector_norm(probe, dim=1)
print("Probe:", tuple(probe.shape), "| norms:", probe_norms.tolist())
assert tuple(probe.shape) == (2, EXPECTED_DIM)
assert bool(torch.isfinite(probe).all()), "Probe có NaN/Inf."
assert torch.allclose(probe_norms, torch.ones_like(probe_norms), atol=1e-3)

bitsandbytes NF4 smoke test: OK


config.json:   0%|          | 0.00/902 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_qwen.py: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model-00007-of-00007.safetensors:   0%|          | 0.00/2.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

Model module: transformers_modules.Alibaba-NLP.gte-Qwen2-7B-instruct.a8d08b36ada9cacfe34c4d6f80957772a025daf2.modeling_qwen
Hidden size: 3584 | rope_theta: 1000000.0
Device map: {'embed_tokens': 0, 'layers.0': 0, 'layers.1': 0, 'layers.2': 0, 'layers.3': 0, 'layers.4': 0, 'layers.5': 0, 'layers.6': 0, 'layers.7': 0, 'layers.8': 0, 'layers.9': 0, 'layers.10': 0, 'layers.11': 1, 'layers.12': 1, 'layers.13': 1, 'layers.14': 1, 'layers.15': 1, 'layers.16': 1, 'layers.17': 1, 'layers.18': 1, 'layers.19': 1, 'layers.20': 1, 'layers.21': 1, 'layers.22': 1, 'layers.23': 1, 'layers.24': 1, 'layers.25': 1, 'layers.26': 1, 'layers.27': 1, 'norm': 1}


2026-06-23 10:16:33.216077: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782209793.450446      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782209793.518313      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782209794.077384      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782209794.077421      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782209794.077424      58 computation_placer.cc:177] computation placer alr

embedded 2 / 2
Probe: (2, 3584) | norms: [1.0, 0.9999999403953552]


## 3. Kiểm tra token, embed document và lưu Qdrant

In [4]:
token_lengths = []
for i, record in enumerate(records, start=1):
    length = len(tokenizer(record["text"], add_special_tokens=True, truncation=False)["input_ids"])
    token_lengths.append(length)
    if i % 500 == 0 or i == len(records):
        print("token-count", i, "/", len(records))

long_indices = [i for i, length in enumerate(token_lengths) if length > MAX_LENGTH]
print(
    "Token min/mean/max:",
    min(token_lengths),
    round(sum(token_lengths) / len(token_lengths), 2),
    max(token_lengths),
)
if long_indices:
    examples = [
        (records[i]["law_id"], records[i]["article_no"], records[i]["aid"], token_lengths[i])
        for i in long_indices[:10]
    ]
    raise ValueError(f"Có {len(long_indices)} article vượt MAX_LENGTH={MAX_LENGTH}: {examples}")

# Document không thêm instruction.
document_vectors = encode_texts([record["text"] for record in records], ENCODE_BATCH)
assert tuple(document_vectors.shape) == (len(records), EXPECTED_DIM)
assert bool(torch.isfinite(document_vectors).all()), "Embedding có NaN/Inf."
norms = torch.linalg.vector_norm(document_vectors, dim=1)
assert torch.allclose(norms, torch.ones_like(norms), atol=1e-3)
print("Document vectors:", tuple(document_vectors.shape), "| norm min/max:", norms.min().item(), norms.max().item())

# Chỉ thay collection sau khi toàn bộ vector đã tạo và kiểm tra thành công.
if qc.collection_exists(COLLECTION):
    qc.delete_collection(COLLECTION)
qc.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=EXPECTED_DIM, distance=Distance.COSINE),
)

for start in range(0, len(records), UPSERT_BATCH):
    end = min(start + UPSERT_BATCH, len(records))
    points = []
    for record, vector in zip(records[start:end], document_vectors[start:end]):
        points.append(PointStruct(
            id=record["point_id"],
            vector=vector.tolist(),
            payload={
                "aid": record["aid"],
                "law_id": record["law_id"],
                "article_no": record["article_no"],
                "content_Article": record["text"],
                "embedding_model": MODEL_ID,
                "embedding_revision": MODEL_REVISION,
                "embedding_pipeline": "custom_gte_last_token_l2_4bit",
                "transformers_version": transformers.__version__,
                "trust_remote_code": True,
                "max_length": MAX_LENGTH,
            },
        ))
    qc.upsert(collection_name=COLLECTION, points=points, wait=True)
    print("upserted", end, "/", len(records))

info = qc.get_collection(COLLECTION)
vectors_config = info.config.params.vectors
stored_dim = int(vectors_config.size)
stored_count = int(qc.count(collection_name=COLLECTION, exact=True).count)
print("Collection:", COLLECTION, "| dim:", stored_dim, "| points:", stored_count)
assert stored_dim == EXPECTED_DIM
assert stored_count == len(records)

token-count 500 / 3352
token-count 1000 / 3352
token-count 1500 / 3352
token-count 2000 / 3352
token-count 2500 / 3352
token-count 3000 / 3352
token-count 3352 / 3352
Token min/mean/max: 19 248.3 2976
embedded 100 / 3352
embedded 200 / 3352
embedded 300 / 3352
embedded 400 / 3352
embedded 500 / 3352
embedded 600 / 3352
embedded 700 / 3352
embedded 800 / 3352
embedded 900 / 3352
embedded 1000 / 3352
embedded 1100 / 3352
embedded 1200 / 3352
embedded 1300 / 3352
embedded 1400 / 3352
embedded 1500 / 3352
embedded 1600 / 3352
embedded 1700 / 3352
embedded 1800 / 3352
embedded 1900 / 3352
embedded 2000 / 3352
embedded 2100 / 3352
embedded 2200 / 3352
embedded 2300 / 3352
embedded 2400 / 3352
embedded 2500 / 3352
embedded 2600 / 3352
embedded 2700 / 3352
embedded 2800 / 3352
embedded 2900 / 3352
embedded 3000 / 3352
embedded 3100 / 3352
embedded 3200 / 3352
embedded 3300 / 3352
embedded 3352 / 3352
Document vectors: (3352, 3584) | norm min/max: 0.9999997019767761 1.000000238418579
upserted 6

## 4. Kiểm tra alignment và retrieval

In [5]:
check_indices = sorted(set([0, len(records) // 2, len(records) - 1]))
for index in check_indices:
    record = records[index]
    hits = qc.query_points(
        collection_name=COLLECTION,
        query=document_vectors[index].tolist(),
        limit=3,
        with_payload=True,
    ).points
    returned_aids = [int((hit.payload or {})["aid"]) for hit in hits]
    assert record["aid"] in returned_aids, f"Self-retrieval fail: aid={record['aid']}"
    print("self-retrieval aid=", record["aid"], "rank=", returned_aids.index(record["aid"]) + 1)

def search(query, top=5):
    query_vector = encode_texts([make_query(query)], 1)[0]
    assert tuple(query_vector.shape) == (EXPECTED_DIM,)
    assert bool(torch.isfinite(query_vector).all())
    hits = qc.query_points(
        collection_name=COLLECTION,
        query=query_vector.tolist(),
        limit=top,
        with_payload=True,
    ).points
    print("\nQuery:", query)
    for rank, hit in enumerate(hits, start=1):
        payload = hit.payload or {}
        snippet = (payload.get("content_Article") or "").replace("\n", " ")[:140]
        print(
            f"{rank:2d}. score={hit.score:.4f} | {payload.get('law_id')} "
            f"Điều {payload.get('article_no')} | aid={payload.get('aid')} | {snippet}..."
        )
    return hits

search("bồi thường thiệt hại do súc vật gây ra", top=5)
search("thời hiệu khởi kiện yêu cầu chia di sản thừa kế", top=5)
print("\n✓ Pipeline embedding và Qdrant đã hoàn tất.")

self-retrieval aid= 270 rank= 1
self-retrieval aid= 53219 rank= 1
self-retrieval aid= 57130 rank= 1
embedded 1 / 1

Query: bồi thường thiệt hại do súc vật gây ra
 1. score=0.8107 | 91/2015/QH13 Điều 603 | aid=53373 | 1. Chủ sở hữu súc vật phải bồi thường thiệt hại do súc vật gây ra cho người khác. Người chiếm hữu, sử dụng súc vật phải bồi thường thiệt hại...
 2. score=0.7274 | 91/2015/QH13 Điều 231 | aid=53001 | 1. Người bắt được gia súc bị thất lạc phải nuôi giữ và báo ngay cho Ủy ban nhân dân cấp xã nơi người đó cư trú để thông báo công khai cho ch...
 3. score=0.6913 | 91/2015/QH13 Điều 232 | aid=53002 | 1. Trường hợp gia cầm của một người bị thất lạc mà người khác bắt được thì người bắt được phải thông báo công khai để chủ sở hữu gia cầm biế...
 4. score=0.6728 | 45/2013/QH13 Điều 90 | aid=56040 | 1. Khi Nhà nước thu hồi đất mà gây thiệt hại đối với cây trồng thì việc bồi thường thực hiện theo quy định sau đây:		a) Đối với cây hàng năm...
 5. score=0.6688 | 91/2015/QH13 Điều 586 | 